# KNN for Economics Science – Solution

**Short name (GitHub):** `KNN_Econ`  
**Lab source:** Codecademy *K-Nearest Neighbors Classifier* adapted to country growth clubs + a macro recession screen  
**Language:** Python (NumPy + pandas + Matplotlib + scikit-learn)

Worked answers. Compare with your `KNN_Econ_Practice_Skeleton.ipynb` cells.  
Companions: `KNN_Econ_Cheatsheet.docx`, `KNN_Econ_Reusable_Template.ipynb`, `knn_econ_flowchart.png`, `KNN_Econ.py`.

### Learning objectives
- Euclidean distance in 2-D and *n*-D on macro ratios
- Min-max scale so inflation (percent) does not drown investment/GDP
- Vote a new country high-growth vs low-growth from its *k* nearest peers
- Fit `KNeighborsClassifier` on a 569-quarter recession book (16 CLI-style series)
- Sweep *k* and plot hold-out accuracy (bias–variance)
- Alternates: NumPy broadcast, Manhattan, `NearestNeighbors`
- Extra practice on a Phillips 2-D plane and a current-account surplus vote
- Simulate *k*, mislabeled quarters, sample size, extra junk series
- Rewrite the finding for an econometrician, a finance-ministry desk, a journalist, a nonspecialist

### Data
- `data/knn_econ_countries.csv` — 40 countries (inflation, I/Y, schooling, high_growth)
- `data/knn_econ_recession.csv` — 569 quarters × 16 series + recession
- `data/knn_econ_2d.csv` — inflation vs Δu plane
- `data/knn_econ_trade.csv` — openness, REER, fiscal, growth gap → CA surplus

### Flowchart
Open `knn_econ_flowchart.png` while you work.


## Inline cheat-sheet (keep this cell visible)

See also **`KNN_Econ_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Euclidean | $d(a,b)=\sqrt{\sum_j (a_j-b_j)^2}$ |
| Manhattan | $d_1(a,b)=\sum_j \|a_j-b_j\|$ |
| Min-max | $x'=(x-x_{\min})/(x_{\max}-x_{\min})$ — fit on **train** |
| Vote | majority class among the $k$ nearest quarters / countries |
| Ties | odd $k$, else nearest neighbor breaks the tie |
| Overfit | $k$ too small → one crisis quarter writes the call |
| Underfit | $k$ too large → vote ≈ sample recession rate |
| sklearn | `KNeighborsClassifier(n_neighbors=k).fit(X,y).score(Xv,yv)` |
| Split | `train_test_split(..., test_size=0.2, random_state=100, stratify=y)` |
| Labels | country `high_growth` 1/0 · quarter `recession` 1/0 |

**Flow:** indicators → scale → split → distance → $k$ peers → vote → sweep $k$ → simulate.


## 0. Packages


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")


## 1. Distance between countries (2-D)


In [ ]:
korea = [2.4, 32.0]
ireland = [1.8, 24.5]
argentina = [42.0, 16.0]

def distance_2d(c1, c2):
    return ((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2) ** 0.5

print(distance_2d(korea, ireland))
print(distance_2d(korea, argentina))
print("closer to Korea:", "Ireland" if distance_2d(korea, ireland) < distance_2d(korea, argentina) else "Argentina")


## 2. Distance in *n* dimensions


In [ ]:
korea_3 = [2.4, 32.0, 14.2]
ireland_3 = [1.8, 24.5, 13.8]
argentina_3 = [42.0, 16.0, 10.6]

def distance(a, b):
    squared = 0.0
    for i in range(len(a)):
        squared += (a[i] - b[i]) ** 2
    return squared ** 0.5

print(distance(korea_3, ireland_3))
print(distance(korea_3, argentina_3))


In [ ]:
def distance_np(a, b):
    return float(np.linalg.norm(np.asarray(a, dtype=float) - np.asarray(b, dtype=float)))

print(distance_np(korea_3, ireland_3))
print("match loop?", np.isclose(distance(korea_3, ireland_3), distance_np(korea_3, ireland_3)))


## 3. Min-max normalization


In [ ]:
inflation = [2.4, 1.8, 42.0, 0.4, 18.0, 80.0, 2.8, 5.4, 3.2, 1.2]

def min_max_normalize(lst):
    lo, hi = min(lst), max(lst)
    return [(v - lo) / (hi - lo) for v in lst]

print(min_max_normalize(inflation))
print("0.4 →", min_max_normalize(inflation)[3], " (lowest π sits at 0); 80 sits at 1")


## 4. Country file + from-scratch classifier


In [ ]:
countries = pd.read_csv("data/knn_econ_countries.csv")
country_dataset = {
    row.country: [float(row.inflation), float(row.investment_gdp), float(row.schooling)]
    for row in countries.itertuples()
}
country_labels = {row.country: int(row.high_growth) for row in countries.itertuples()}
print("n countries:", len(country_dataset))
print("sample:", list(country_dataset.items())[:2])


In [ ]:
def fit_minmax(dataset):
    X = np.array(list(dataset.values()), dtype=float)
    return X.min(axis=0), X.max(axis=0)

def apply_minmax(point, mins, maxs):
    point = np.asarray(point, dtype=float)
    span = np.where(maxs - mins == 0, 1.0, maxs - mins)
    return ((point - mins) / span).tolist()

def normalize_dataset(dataset):
    mins, maxs = fit_minmax(dataset)
    return {t: apply_minmax(v, mins, maxs) for t, v in dataset.items()}, mins, maxs

country_dataset_n, mins, maxs = normalize_dataset(country_dataset)
print("mins:", mins)
print("maxs:", maxs)


In [ ]:
def classify(unknown, dataset, labels, k):
    distances = [[distance(unknown, point), name] for name, point in dataset.items()]
    distances.sort()
    neighbors = distances[:k]
    n_high = sum(1 for _, name in neighbors if labels[name] == 1)
    return 1 if n_high > k / 2 else 0


In [ ]:
name = "Canada"
print("in set?", name in country_dataset)
held = {t: p for t, p in country_dataset_n.items() if t != name}
held_y = {t: lab for t, lab in country_labels.items() if t != name}
unknown_n = country_dataset_n[name]
pred = classify(unknown_n, held, held_y, 5)
print("normalized Canada:", unknown_n)
print("k=5 vote (1=high growth):", pred, "  actual:", country_labels[name])


## 5. Alternate neighbor search


In [ ]:
def classify_np(unknown, X, y, k):
    d = np.linalg.norm(X - np.asarray(unknown, dtype=float), axis=1)
    idx = np.argsort(d)[:k]
    return int(y[idx].sum() > k / 2)

names = list(country_dataset_n.keys())
X_c = np.array([country_dataset_n[t] for t in names])
y_c = np.array([country_labels[t] for t in names])
print("numpy vote:", classify_np(unknown_n, X_c, y_c, 5))

nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn.fit(X_c)
_, ind = nn.kneighbors([unknown_n])
print("sklearn peers:", [names[i] for i in ind[0]])


In [ ]:
def classify_manhattan(unknown, X, y, k):
    d = np.abs(X - np.asarray(unknown, dtype=float)).sum(axis=1)
    idx = np.argsort(d)[:k]
    return int(y[idx].sum() > k / 2)

print("Manhattan vote:", classify_manhattan(unknown_n, X_c, y_c, 5))


## 6. Recession book


In [ ]:
rec = pd.read_csv("data/knn_econ_recession.csv")
print("shape:", rec.shape)
print("columns:", list(rec.columns))
print(rec.head(2))
print("recession count:", int(rec.recession.sum()), "expansion:", int((1 - rec.recession).sum()))


In [ ]:
print("sample recession rate:", rec.recession.mean())
print("naive always-expansion accuracy would be", 1 - rec.recession.mean())


## 7. Train / validation split


In [ ]:
y = rec["recession"].to_numpy()
X = rec.drop(columns=["recession"]).to_numpy()
training_data, validation_data, training_labels, validation_labels = train_test_split(
    X, y, test_size=0.2, random_state=100, stratify=y
)
print("train", len(training_data), "valid", len(validation_data))
print("lengths match?", len(training_data) == len(training_labels))


## 8. KNeighborsClassifier


In [ ]:
clf_raw = KNeighborsClassifier(n_neighbors=3)
clf_raw.fit(training_data, training_labels)
print("k=3 unscaled valid acc:", clf_raw.score(validation_data, validation_labels))


In [ ]:
scaler = MinMaxScaler()
Xtr_s = scaler.fit_transform(training_data)
Xva_s = scaler.transform(validation_data)
clf_s = KNeighborsClassifier(n_neighbors=3)
clf_s.fit(Xtr_s, training_labels)
print("k=3 scaled valid acc:", clf_s.score(Xva_s, validation_labels))


## 9. Sweep *k* and graph


In [ ]:
k_list = list(range(1, 51))
accuracies = []
for k in k_list:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(Xtr_s, training_labels)
    accuracies.append(clf.score(Xva_s, validation_labels))
best_i = int(np.argmax(accuracies))
print("best k, acc:", k_list[best_i], accuracies[best_i])
print("k=1:", accuracies[0], " k=3:", accuracies[2], " k=15:", accuracies[14])


In [ ]:
plt.figure(figsize=(8.4, 4.2))
plt.plot(k_list, accuracies, color="#1A5276")
plt.axvline(k_list[best_i], color="#117A65", ls="--", label=f"best k={k_list[best_i]}")
plt.xlabel("k")
plt.ylabel("Validation Accuracy")
plt.title("Recession Classifier Accuracy")
plt.legend()
plt.show()


## 10. More practice


In [ ]:
plane = pd.read_csv("data/knn_econ_2d.csv")
Xb = plane[["inflation", "unemp_change"]].to_numpy()
yb = plane["recession"].to_numpy()
Xtrb, Xvab, ytrb, yvab = train_test_split(Xb, yb, test_size=0.2, random_state=11, stratify=yb)

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))
xx, yy = np.meshgrid(
    np.linspace(Xb[:, 0].min() - 0.6, Xb[:, 0].max() + 0.6, 200),
    np.linspace(Xb[:, 1].min() - 0.25, Xb[:, 1].max() + 0.25, 200),
)
grid = np.c_[xx.ravel(), yy.ravel()]
for ax, k in zip(axes, [1, 21]):
    clf = KNeighborsClassifier(n_neighbors=k).fit(Xtrb, ytrb)
    zz = clf.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=["#D6EAF8", "#FADBD8"])
    ax.scatter(Xtrb[ytrb == 0, 0], Xtrb[ytrb == 0, 1], c="#1A5276", s=14)
    ax.scatter(Xtrb[ytrb == 1, 0], Xtrb[ytrb == 1, 1], c="#C0392B", s=14)
    ax.set_title(f"k={k}  valid acc={clf.score(Xvab, yvab):.3f}")
    ax.set_xlabel("inflation"); ax.set_ylabel("Δu")
plt.suptitle("Phillips-plane practice: small k hugs every quarter")
plt.tight_layout()
plt.show()


In [ ]:
tr = pd.read_csv("data/knn_econ_trade.csv")
Xl = tr[["trade_openness", "reer", "fiscal_balance", "growth_gap"]].to_numpy()
yl = tr["ca_surplus"].to_numpy()
Xtrl, Xval, ytrl, yval = train_test_split(Xl, yl, test_size=0.25, random_state=21, stratify=yl)
sc_l = MinMaxScaler()
Xtrl_s, Xval_s = sc_l.fit_transform(Xtrl), sc_l.transform(Xval)
for k in (1, 5, 15):
    clf = KNeighborsClassifier(n_neighbors=k).fit(Xtrl_s, ytrl)
    pred = clf.predict(Xval_s)
    print(f"k={k:2d}  acc={accuracy_score(yval, pred):.3f}  cm={confusion_matrix(yval, pred).tolist()}")


## 11. Simulation


In [ ]:
# ----- editable -----
K_FIXED = 3
NOISE = 0.00
TRAIN_FRAC = 1.00
N_EXTRA = 0
RANDOM_STATE = 100
# --------------------

rec = pd.read_csv("data/knn_econ_recession.csv")
y = rec["recession"].to_numpy()
X = rec.drop(columns=["recession"]).to_numpy()
Xtr, Xva, ytr, yva = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
sc = MinMaxScaler()
Xtr, Xva = sc.fit_transform(Xtr), sc.transform(Xva)

rng = np.random.default_rng(RANDOM_STATE)
m = max(K_FIXED + 1, int(len(Xtr) * TRAIN_FRAC))
idx = rng.choice(len(Xtr), size=m, replace=False)
Xtr, ytr = Xtr[idx], ytr[idx].copy()
flip = rng.random(len(ytr)) < NOISE
ytr[flip] = 1 - ytr[flip]
if N_EXTRA > 0:
    Xtr = np.hstack([Xtr, rng.normal(size=(len(Xtr), N_EXTRA))])
    Xva = np.hstack([Xva, rng.normal(size=(len(Xva), N_EXTRA))])
    sc2 = MinMaxScaler()
    Xtr, Xva = sc2.fit_transform(Xtr), sc2.transform(Xva)

clf = KNeighborsClassifier(n_neighbors=min(K_FIXED, len(Xtr)))
clf.fit(Xtr, ytr)
print(f"valid acc = {clf.score(Xva, yva):.4f}   (k={K_FIXED}, noise={NOISE}, n_train={len(Xtr)}, extra={N_EXTRA})")
print("try: K_FIXED=1, NOISE=0.2, TRAIN_FRAC=0.2, N_EXTRA=80")


## 12. Audience rewrite


In [ ]:
analyst = (
    "Min-max Euclidean KNN on 16 CLI-style series, 80/20 stratified split "
    "(random_state=100), scores 0.947 at k=3 on 114 hold-out quarters versus 0.754 unscaled. "
    "k=1 is twitchier; padding 80 N(0,1) columns drops accuracy toward 0.85 "
    "(curse of dimensionality). Fit the scaler on train only. Report sensitivity to k, "
    "label noise, and n_train before locking a neighbor count. This is a nonparametric "
    "screen, not a structural VAR."
)
desk = (
    "A nearest-neighbor screen that lines up the current quarter against the most similar "
    "past quarters tagged expansion versus recession correctly on about 95 of 100 leftover "
    "quarters when the series are put on a 0–1 scale (k = 3). Raw mixed units only reached "
    "the mid-70s. It is a nowcast aid sitting next to the official dating committee, not a "
    "substitute for it. Dumping extra unused series makes ‘similar’ harder to define."
)
journalist = (
    "Researchers compared each new quarter with a handful of past quarters that looked "
    "most like it on things such as industrial output, the yield curve and unemployment. "
    "On a set-aside test the vote matched the recorded expansion/recession label about "
    "95% of the time after putting every series on the same scale. It is one extra check, "
    "not an official recession call."
)
friend = (
    "Imagine lining up past quarters that look most like this one and taking a majority "
    "vote on whether those twins were recessions. With three neighbors, after stretching "
    "every chart onto the same 0–1 ruler, the vote was right about 95 times out of 100 "
    "on leftover data. Too few neighbors copies flukes; too many just repeats the usual "
    "‘most quarters are expansions’ answer."
)
print(analyst); print(); print(desk); print(); print(journalist); print(); print(friend)


## Done

Reusable pattern: `KNN_Econ_Reusable_Template.ipynb`. Charts: `knn_econ_*.png`.
